## Setup

### Libraries

In [9]:
import os
import glob
import numpy as np
import pandas as pd
from collections import defaultdict
import pickle

### Data Extraction Functions

In [10]:
def load_lift_drag_data(root_dir):
    data_by_config_aoa = defaultdict(lambda: {'lift': [], 'drag': [], 'turbulence_model': '', 'geometry': '', 'mesh': '', 'version': '', 'grid': ''})
    
    for dirpath, _, filenames in os.walk(root_dir):
        # Find lift and drag files
        lift_files = [f for f in filenames if 'lift_force' in f and f.endswith('.txt')]
        drag_files = [f for f in filenames if 'drag_force' in f and f.endswith('.txt')]
        
        if lift_files and drag_files:
            config = None
            
            # Extract configuration based on method
            if CONFIG_EXTRACTION_METHOD == 'case_file':
                # Look for .cas or .cas.h5 files and extract filename
                for filename in filenames:
                    if filename.endswith('.cas') or filename.endswith('.cas.h5'):
                        config = filename.replace('.cas.h5', '').replace('.cas', '')
                        if config and config[0].isdigit() and '.' in config:
                            break
                        else:
                            config = None
            else:
                # Legacy: parse from folder structure
                parts = dirpath.split(os.sep)
                for part in parts:
                    if part and part[0].isdigit() and part.count('.') >= 2:
                        config = part
                        break
            
            # Find AoA value from folder name (e.g., AoA_10)
            aoa = None
            parts = dirpath.split(os.sep)
            for part in parts:
                if part.startswith('AoA_'):
                    aoa = part
                    break
            
            if config and aoa:
                # Extract AoA number
                aoa_number = aoa.split('_')[1]
                
                # Check if config contains _AoA_ pattern (old format embedded in filename)
                # Example: "4.3.1.3_AoA_5" becomes "4.3.1.3.5"
                if '_AoA_' in config:
                    config = config.replace(f'_AoA_{aoa_number}', f'.{aoa_number}')
                # Check if this is old format with no dots (version name only)
                # Example: "version_name" becomes "version_name.5"
                elif config.count('.') == 0:
                    config = f"{config}.{aoa_number}"
                
                # Parse configuration using position and value mappings
                config_parts = config.split('.')
                positions = POSITION_MAP
                mappings = VALUE_MAPPINGS
                
                # Extract each field based on position
                geometry_num = config_parts[positions['geometry']] if positions['geometry'] is not None and len(config_parts) > positions['geometry'] else None
                mesh_num = config_parts[positions['mesh']] if positions['mesh'] is not None and len(config_parts) > positions['mesh'] else None
                turbulence_num = config_parts[positions['turbulence']] if positions['turbulence'] is not None and len(config_parts) > positions['turbulence'] else None
                version_num = config_parts[positions['version']] if positions['version'] is not None and len(config_parts) > positions['version'] else None
                grid_code = config_parts[positions['grid']] if positions['grid'] is not None and len(config_parts) > positions['grid'] else None
                
                # Map to descriptive names
                geometry = mappings.get('geometry', {}).get(geometry_num, f"Geometry_{geometry_num}") if geometry_num else "N/A"
                mesh = mappings.get('mesh', {}).get(mesh_num, f"Mesh_{mesh_num}") if mesh_num else "N/A"
                turbulence_model = mappings.get('turbulence', {}).get(turbulence_num, f"Turbulence_{turbulence_num}") if turbulence_num else "Unknown"
                version = mappings.get('version', {}).get(version_num, f"Version_{version_num}") if version_num else "N/A"
                grid = mappings.get('grid', {}).get(grid_code, f"Grid_{grid_code}") if grid_code else "N/A"
                
                # Extract angle of attack in degrees (e.g., AoA_10 → 10)
                aoa_degrees = float(aoa_number)
                aoa_radians = np.radians(aoa_degrees)
                
                # Read all lift files
                for lift_file in lift_files:
                    lift_path = os.path.join(dirpath, lift_file)
                    with open(lift_path) as f:
                        lift_data = []
                        for line in f:
                            line = line.strip()
                            # Skip header lines (lines with quotes or non-numeric content)
                            if not line or '"' in line or '(' in line:
                                continue
                            # Split by whitespace and take the second column
                            parts_line = line.split()
                            if len(parts_line) >= 2:
                                try:
                                    value = float(parts_line[1])
                                    lift_data.append(value)
                                except ValueError:
                                    continue
                
                # Read all drag files
                for drag_file in drag_files:
                    drag_path = os.path.join(dirpath, drag_file)
                    with open(drag_path) as f:
                        drag_data = []
                        for line in f:
                            line = line.strip()
                            # Skip header lines (lines with quotes or non-numeric content)
                            if not line or '"' in line or '(' in line:
                                continue
                            # Split by whitespace and take the second column
                            parts_line = line.split()
                            if len(parts_line) >= 2:
                                try:
                                    value = float(parts_line[1])
                                    drag_data.append(value)
                                except ValueError:
                                    continue
                
                # Apply angle of attack correction to transform forces
                # True lift = lift*cos(theta) - drag*sin(theta)
                # True drag = lift*sin(theta) + drag*cos(theta)
                cos_theta = np.cos(aoa_radians)
                sin_theta = np.sin(aoa_radians)
                
                true_lift_data = []
                true_drag_data = []
                
                # Ensure both arrays have the same length
                min_length = min(len(lift_data), len(drag_data))
                for i in range(min_length):
                    true_lift = lift_data[i] * cos_theta - drag_data[i] * sin_theta
                    true_drag = lift_data[i] * sin_theta + drag_data[i] * cos_theta
                    true_lift_data.append(true_lift)
                    true_drag_data.append(true_drag)
                
                # Store the corrected data
                data_by_config_aoa[(config, aoa)]['lift'].extend(true_lift_data)
                data_by_config_aoa[(config, aoa)]['drag'].extend(true_drag_data)
                
                # Store metadata
                data_by_config_aoa[(config, aoa)]['geometry'] = geometry
                data_by_config_aoa[(config, aoa)]['mesh'] = mesh
                data_by_config_aoa[(config, aoa)]['turbulence_model'] = turbulence_model
                data_by_config_aoa[(config, aoa)]['version'] = version
                data_by_config_aoa[(config, aoa)]['grid'] = grid
    
    return data_by_config_aoa

In [11]:
def compute_statistics(data):
    mean_val = np.mean(data)
    std_dev = np.std(data)
    # Calculate Coefficient of Variation (COV) as percentage
    cov = (std_dev / mean_val * 100) if mean_val != 0 else 0
    return mean_val, cov

## User Inputs - Modify These

In [12]:
# ==================== USER INPUTS - MODIFY THESE ====================

# Input data directory
base_path = r"C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Thesis\NACA_2414_2D\Fleunt\Directories\2414_006_005\3.1.1.NG"

# Output directory for results
output_dir = r"C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Thesis\NACA_2414_2D\Fleunt\Directories\2414_006_005\3.1.1.NG"

# Configuration extraction method
# Options: 'case_file' (extract from .cas filename) or 'folder' (extract from folder structure)
CONFIG_EXTRACTION_METHOD = 'case_file'

# Position mapping - which position in the config string corresponds to which field
# Format: config string is "geometry.mesh.turbulence.version.grid.aoa"
# Example: "4.3.1.2.NG.5" means geometry=4, mesh=3, turbulence=1, version=2, grid=NG, aoa=5
POSITION_MAP = {
    'geometry': None,      # Position 0: Geometry number
    'mesh': 1,          # Position 1: Mesh number
    'turbulence': 2,    # Position 2: Turbulence model number
    'version': 3,       # Position 3: Version number
    'grid': 4,          # Position 4: Grid type (e.g., NG)
    # Note: AoA is always the last position
}

# Value mappings - convert numbers/codes to descriptive names
VALUE_MAPPINGS = {
    'geometry': {
        3: '2414_006_002',
        4: '2414_006_005',
    },
    'mesh': {
        1: 'Coarse',
        2: 'Medium',
        3: 'Fine',
        4: 'ExtraFine',
        5: 'Ultra',
    },
    'turbulence': {
        1: 'SST',
        2: 'RNG',
        3: 'RSM',
    },
    'version': {
        1: 'V1',
        2: 'V2',
        3: 'V3',
    },
    'grid': {
        'NG': 'No Grid',
        'G': 'With Grid',
    }
}

# Turbulence model ordering for sorting in outputs
TURBULENCE_ORDER = {
    'k-epsilon': 1,
    'k-omega': 2,
    'Transition SST': 3,
}

# Comparison configurations for turbulence model comparison sheets
# Format: 'config_name': ['list', 'of', 'configs']
COMPARISON_CONFIGS = {
    '4.3.NG': ['4.3.1.NG', '4.3.2.NG', '4.3.3.NG'],  # Fine mesh comparison
}

# =====================================================================

## Load and Process Data

In [13]:
# Load all data
print("Loading lift and drag data from:", base_path)
all_data = load_lift_drag_data(base_path)

print(f"\nLoaded data for {len(all_data)} configuration-AoA combinations:")
for (config, aoa), data in sorted(all_data.items()):
    print(f"  {config} @ {aoa}: {len(data['lift'])} data points - {data['turbulence_model']}")

print("\n✓ Data loading complete!")
print(f"Total configurations found: {len(set(config for config, aoa in all_data.keys()))}")
print(f"Total AoA values found: {len(set(aoa for config, aoa in all_data.keys()))}")

Loading lift and drag data from: C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Thesis\NACA_2414_2D\Fleunt\Directories\2414_006_005\3.1.1.NG

Loaded data for 16 configuration-AoA combinations:
  3.1.1.NG.10 @ AoA_10: 1200 data points - Turbulence_1
  3.1.1.NG.11 @ AoA_11: 1200 data points - Turbulence_1
  3.1.1.NG.12 @ AoA_12: 1200 data points - Turbulence_1
  3.1.1.NG.13 @ AoA_13: 1200 data points - Turbulence_1
  3.1.1.NG.14 @ AoA_14: 1200 data points - Turbulence_1
  3.1.1.NG.15 @ AoA_15: 1200 data points - Turbulence_1
  3.1.1.NG.16 @ AoA_16: 1200 data points - Turbulence_1
  3.1.1.NG.17 @ AoA_17: 1200 data points - Turbulence_1
  3.1.1.NG.18 @ AoA_18: 1200 data points - Turbulence_1
  3.1.1.NG.19 @ AoA_19: 1200 data points - Turbulence_1
  3.1.1.NG.20 @ AoA_20: 1200 data points - Turbulence_1
  3.1.1.NG.5 @ AoA_5: 1200 data points - Turbulence_1
  3.1.1.NG.6 @ AoA_6: 1200 data points - Turbulence_1
  3.1.1.NG.7 @ AoA_7: 1200 data points - Turbulence_1
  3.1.1.NG.8 @ AoA_8: 1200 data 

## Save Processed Data

In [14]:
# Save the processed data to pickle file for use in other notebooks
pickle_file = os.path.join(output_dir, 'processed_data.pkl')

# Convert defaultdict to regular dict for pickling
all_data_dict = dict(all_data)

with open(pickle_file, 'wb') as f:
    pickle.dump({
        'all_data': all_data_dict,
        'config_info': {
            'POSITION_MAP': POSITION_MAP,
            'VALUE_MAPPINGS': VALUE_MAPPINGS,
            'TURBULENCE_ORDER': TURBULENCE_ORDER,
            'COMPARISON_CONFIGS': COMPARISON_CONFIGS,
            'CONFIG_EXTRACTION_METHOD': CONFIG_EXTRACTION_METHOD,
        },
        'paths': {
            'base_path': base_path,
            'output_dir': output_dir,
        }
    }, f)

print(f"✓ Data saved to: {pickle_file}")
print(f"  File size: {os.path.getsize(pickle_file) / 1024:.1f} KB")
print("\nNext step: Run notebook 2_convergence_analysis.ipynb (optional) or 3_excel_outputs.ipynb")

✓ Data saved to: C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Thesis\NACA_2414_2D\Fleunt\Directories\2414_006_005\3.1.1.NG\processed_data.pkl
  File size: 715.4 KB

Next step: Run notebook 2_convergence_analysis.ipynb (optional) or 3_excel_outputs.ipynb


## Create Summary Statistics Text File

## Export Processed Data to Text Files

In [15]:
# Export processed data to individual text files
processed_data_dir = os.path.join(output_dir, "processed_data")
os.makedirs(processed_data_dir, exist_ok=True)

print("\n" + "=" * 80)
print("EXPORTING PROCESSED DATA TO TEXT FILES")
print("=" * 80)

# Export each dataset with a unique name based on config (config includes AoA)
for (config, aoa), data in all_data_dict.items():
    # Use config as filename base since it already includes the AoA (e.g., "4.3.1.3.NG.5")
    filename_base = config
    
    # Export lift data
    lift_output = os.path.join(processed_data_dir, f"{filename_base}_lift.txt")
    with open(lift_output, 'w', encoding='utf-8') as f:
        f.write(f"# Processed Lift Data\n")
        f.write(f"# Configuration: {config}\n")
        f.write(f"# Angle of Attack: {aoa}\n")
        f.write(f"# Turbulence Model: {data['turbulence_model']}\n")
        f.write(f"# Geometry: {data['geometry']}\n")
        f.write(f"# Mesh: {data['mesh']}\n")
        f.write(f"# Grid: {data['grid']}\n")
        f.write(f"# Total Points: {len(data['lift'])}\n")
        f.write(f"# Note: AoA correction applied (True Lift = Lift*cos(θ) - Drag*sin(θ))\n")
        f.write("#\n")
        for value in data['lift']:
            f.write(f"{value}\n")
    
    # Export drag data
    drag_output = os.path.join(processed_data_dir, f"{filename_base}_drag.txt")
    with open(drag_output, 'w', encoding='utf-8') as f:
        f.write(f"# Processed Drag Data\n")
        f.write(f"# Configuration: {config}\n")
        f.write(f"# Angle of Attack: {aoa}\n")
        f.write(f"# Turbulence Model: {data['turbulence_model']}\n")
        f.write(f"# Geometry: {data['geometry']}\n")
        f.write(f"# Mesh: {data['mesh']}\n")
        f.write(f"# Grid: {data['grid']}\n")
        f.write(f"# Total Points: {len(data['drag'])}\n")
        f.write(f"# Note: AoA correction applied (True Drag = Lift*sin(θ) + Drag*cos(θ))\n")
        f.write("#\n")
        for value in data['drag']:
            f.write(f"{value}\n")

print(f"\n✓ Processed data exported to: {processed_data_dir}")
print(f"✓ Total files created: {len(all_data_dict) * 2} ({len(all_data_dict)} configs × 2 files)")
print("=" * 80)



EXPORTING PROCESSED DATA TO TEXT FILES

✓ Processed data exported to: C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Thesis\NACA_2414_2D\Fleunt\Directories\2414_006_005\3.1.1.NG\processed_data
✓ Total files created: 32 (16 configs × 2 files)

✓ Processed data exported to: C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Thesis\NACA_2414_2D\Fleunt\Directories\2414_006_005\3.1.1.NG\processed_data
✓ Total files created: 32 (16 configs × 2 files)


In [16]:
# Create summary statistics text file with basic statistics
summary_file = os.path.join(output_dir, "SUMMARY_Statistics.txt")
num_iterations = 150  # Number of last iterations to use for statistics

with open(summary_file, 'w', encoding='utf-8') as f:
    f.write("=" * 100 + "\n")
    f.write(f"DATA PROCESSING SUMMARY - Last {num_iterations} Iterations\n")
    f.write("=" * 100 + "\n\n")
    f.write(f"Date: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Base Path: {base_path}\n")
    f.write(f"Output Directory: {output_dir}\n")
    f.write(f"Extraction Method: {CONFIG_EXTRACTION_METHOD}\n\n")
    f.write("=" * 100 + "\n\n")
    
    # Sort by config and then by AoA numerically
    def extract_aoa_number(aoa_str):
        try:
            return int(aoa_str.split('_')[1])
        except:
            return 0
    
    sorted_data = sorted(all_data_dict.items(), key=lambda x: (x[0][0], extract_aoa_number(x[0][1])))
    
    for (config, aoa), data in sorted_data:
        # Get last N iterations (or all if less than N)
        lift_last_n = data['lift'][-num_iterations:] if len(data['lift']) >= num_iterations else data['lift']
        drag_last_n = data['drag'][-num_iterations:] if len(data['drag']) >= num_iterations else data['drag']
        
        # Calculate statistics
        lift_mean, lift_cov = compute_statistics(lift_last_n) if lift_last_n else (0, 0)
        drag_mean, drag_cov = compute_statistics(drag_last_n) if drag_last_n else (0, 0)
        
        # Write to file
        f.write(f"Configuration: {config}\n")
        f.write(f"  Turbulence Model: {data['turbulence_model']}\n")
        f.write(f"  Geometry: {data['geometry']}\n")
        f.write(f"  Mesh: {data['mesh']}\n")
        f.write(f"  Grid: {data['grid']}\n")
        f.write(f"  Angle of Attack: {aoa}\n")
        f.write(f"  Total Data Points: {len(data['lift'])}\n")
        f.write(f"  Points Used: {len(lift_last_n)}\n")
        f.write("-" * 100 + "\n")
        f.write(f"  Lift Mean:  {lift_mean:12.4f} N\n")
        f.write(f"  Lift COV:   {lift_cov:12.2f} %\n")
        f.write(f"  Drag Mean:  {drag_mean:12.4f} N\n")
        f.write(f"  Drag COV:   {drag_cov:12.2f} %\n")
        f.write("=" * 100 + "\n\n")

print(f"✓ Summary statistics written to: {summary_file}")
print(f"  {len(all_data_dict)} configurations summarized")


✓ Summary statistics written to: C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Thesis\NACA_2414_2D\Fleunt\Directories\2414_006_005\3.1.1.NG\SUMMARY_Statistics.txt
  16 configurations summarized
